# SepsisGuard — GRPO Training Notebook

Train 4 multi-agent roles (Nurse, Lab, Pharmacist, Physician) using TRL GRPO.

In [ ]:
# Cell 1 — Install
!pip install -q -U unsloth openenv-core trl>=0.12 vllm datasets
!pip install -q requests httpx

In [ ]:
# Cell 2 — Load model
from unsloth import FastLanguageModel
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
# Cell 3 — Connect to env
import os, requests
ENV_URL = os.environ.get("ENV_BASE_URL", "https://YOUR-USERNAME-sepsisguard.hf.space")

class EnvClient:
    def reset(self, task_name, seed):
        r = requests.post(f"{ENV_URL}/reset",
                          json={"task_name": task_name, "seed": seed}, timeout=30)
        r.raise_for_status(); return r.json()
    def step(self, actions):
        r = requests.post(f"{ENV_URL}/step", json={"actions": actions}, timeout=30)
        r.raise_for_status(); return r.json()

env = EnvClient()
print(env.reset(task_name="task1_textbook", seed=42)["info"])

In [ ]:
# Cell 4 — Collect rollouts
import sys; sys.path.insert(0, "/content/sepsisguard")
from training.rollout_collector import collect_rollouts
rollouts = collect_rollouts(model, tokenizer, env, n_episodes=2, task="task1_textbook")
print(f"Collected {len(rollouts)} rollout steps")

In [ ]:
# Cell 5 — GRPO Training
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from training.reward_shaping import make_online_sepsis_reward_fn

FastLanguageModel.for_training(model)

train_dataset = Dataset.from_list([
    {"prompt": r["prompt"],
     "metadata": {"env_reward_placeholder": r["env_reward_placeholder"]}}
    for r in rollouts
])

cfg = GRPOConfig(
    output_dir="./sepsis-gen1",
    num_generations=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_steps=50,
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=50,
    max_prompt_length=3000,
    max_completion_length=96,
)

# Live reward: evaluate each generated completion with a real /reset + /step call.
reward_fn = make_online_sepsis_reward_fn(
    env_url=ENV_URL,
    task_name="task1_textbook",
    seed=42,
)

trainer = GRPOTrainer(
    model=model, reward_funcs=[reward_fn],
    args=cfg, train_dataset=train_dataset,
)
trainer.train()

In [ ]:
# Cell 6 — Save checkpoint
model.save_pretrained("./sepsis-gen1")
tokenizer.save_pretrained("./sepsis-gen1")
print("Checkpoint saved!")